# Banana Ripeness — ConvNeXt Embeddings

Download the Banana Ripeness Classification dataset, stage it locally, and extract ConvNeXt backbone embeddings.

In [1]:
import shutil
from pathlib import Path

import kagglehub as kh
import torch
from PIL import Image
from rembg import new_session, remove
from torchvision.models import ConvNeXt_Tiny_Weights, convnext_tiny

In [2]:
# Config
DATASET_SLUG = "shahriar26s/banana-ripeness-classification-dataset"
DATASET_DIR_NAME = "Banana Ripeness Classification Dataset"
WORK_DIR = Path("/tmp/bananafp")
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEDUPED_DIR_NAME = "deduped"
DEDUPED_DIR = WORK_DIR / DEDUPED_DIR_NAME
NOBG_DIR_NAME = "nobg"
NOBG_DIR = WORK_DIR / NOBG_DIR_NAME
NN_SIM_THRESHOLD = 0.98  # cosine sim >= thresh -> duplicate
N_CLUSTERS = 4  # overripe / ripe / rotten / unripe

In [3]:
# Download dataset (cached by kagglehub)
cache_path = Path(kh.dataset_download(DATASET_SLUG))
print(f"Dataset cached at: {cache_path}")

Dataset cached at: /home/irzam/.cache/kagglehub/datasets/shahriar26s/banana-ripeness-classification-dataset/versions/1


In [4]:
# Stage to work dir and collect image paths
src = cache_path / DATASET_DIR_NAME if (cache_path / DATASET_DIR_NAME).exists() else cache_path
dataset = WORK_DIR / DATASET_DIR_NAME

if not dataset.exists():
    shutil.copytree(src, dataset)

image_paths = sorted(p for p in dataset.rglob("*") if p.suffix.lower() in IMAGE_EXTS)
print(f"{len(image_paths)} images in {dataset}")
print(image_paths[:5])

13478 images in /tmp/bananafp/Banana Ripeness Classification Dataset
[PosixPath('/tmp/bananafp/Banana Ripeness Classification Dataset/test/overripe/musa-acuminata-mold-e18cfd23-1d0a-11ec-87d5-d8c4975e38aa_jpg.rf.175dccdf3479ad6a73213197efe00527.jpg'), PosixPath('/tmp/bananafp/Banana Ripeness Classification Dataset/test/overripe/musa-acuminata-mold-e1a4d2b8-1d0a-11ec-af1f-d8c4975e38aa_jpg.rf.49b2874813b9606a2e643b694212cfc1.jpg'), PosixPath('/tmp/bananafp/Banana Ripeness Classification Dataset/test/overripe/musa-acuminata-mold-e1ae5b43-1d0a-11ec-80fd-d8c4975e38aa_jpg.rf.e3757cdfe9a0f7a6a659b6e6858e3a2a.jpg'), PosixPath('/tmp/bananafp/Banana Ripeness Classification Dataset/test/overripe/musa-acuminata-mold-e1de067f-1d0a-11ec-bae6-d8c4975e38aa_jpg.rf.1ab7aa2244f9fe003f316216283f5844.jpg'), PosixPath('/tmp/bananafp/Banana Ripeness Classification Dataset/test/overripe/musa-acuminata-mold-e1fd0281-1d0a-11ec-801c-d8c4975e38aa_jpg.rf.cdfd103ba1477ff7acd004cf2f6cdef3.jpg')]


In [5]:
# ConvNeXt backbone only (strip classification head -> embedding)
weights = ConvNeXt_Tiny_Weights.DEFAULT
preprocess = weights.transforms()
model = convnext_tiny(weights=weights)
model.classifier[2] = torch.nn.Identity()  # (768,) embedding instead of 1000-way logits
model = model.eval().to(DEVICE)

@torch.no_grad()
def embed(pil_images, batch_size: int = 32):  # list[PIL.Image] -> (N, 768)
    """Embed a list of RGB PIL images with the ConvNeXt backbone, batched to avoid OOM."""
    feats = []
    for i in range(0, len(pil_images), batch_size):
        batch = torch.stack([preprocess(img.convert("RGB")) for img in pil_images[i : i + batch_size]]).to(DEVICE)
        feats.append(model(batch).cpu())
    return torch.cat(feats).numpy() if feats else None

## Pipeline: dataset -> embedding -> dedupe (NN) -> remove bg -> re-embedding -> KMeans

1. Embed raw images with the ConvNeXt backbone.
2. NN dedupe per `split/class`: greedily keep an image only if its max cosine similarity to kept ones is below `NN_SIM_THRESHOLD`. Copies keepers to `DEDUPED_DIR`.
3. Remove background (`rembg`, white composite) of deduped images only, cached in `NOBG_DIR`.
4. Re-embed the bg-removed images.
5. KMeans (`k=4`) on train re-embeddings as the final model; map clusters to labels by majority vote and evaluate on valid/test.

In [6]:
import numpy as np
from sklearn.cluster import KMeans

In [7]:
# Embed image paths in batches, streaming from disk
def embed_paths(paths, batch_size: int = 32):
    vecs = []
    for i in range(0, len(paths), batch_size):
        batch = [Image.open(p).convert("RGB") for p in paths[i : i + batch_size]]
        vecs.append(embed(batch))
    return np.concatenate(vecs) if vecs else None

In [8]:
# NN dedupe: keep image iff max cosine sim to kept set < threshold (per split/class)
def nn_keepers(embs: np.ndarray, thresh: float = NN_SIM_THRESHOLD):
    Z = embs / np.linalg.norm(embs, axis=1, keepdims=True).clip(min=1e-9)
    kept_idx, dup_idx = [], []
    for i in range(len(Z)):
        if kept_idx and (Z[i] @ Z[kept_idx].T).max() >= thresh:
            dup_idx.append(i)
        else:
            kept_idx.append(i)
    return kept_idx, dup_idx

SPLITS_DEDUPE = ["train"]  # dedupe train; extend if needed

if DEDUPED_DIR.exists():
    shutil.rmtree(DEDUPED_DIR)

for split in SPLITS_DEDUPE:
    for class_dir in sorted((dataset / split).iterdir()):
        if not class_dir.is_dir():
            continue
        paths = sorted(p for p in class_dir.rglob("*") if p.suffix.lower() in IMAGE_EXTS)
        embs = embed_paths(paths)
        keep, dup = nn_keepers(embs)
        for k in keep:
            dst = DEDUPED_DIR / split / class_dir.name / paths[k].name
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(paths[k], dst)
        print(f"{split}/{class_dir.name}: {len(paths)} -> {len(keep)} kept ({len(dup)} dups)")

print("deduped at:", DEDUPED_DIR)

train/overripe: 2349 -> 2079 kept (270 dups)
train/ripe: 3522 -> 3086 kept (436 dups)
train/rotten: 4020 -> 3875 kept (145 dups)
train/unripe: 1902 -> 1690 kept (212 dups)
deduped at: /tmp/bananafp/deduped


In [9]:
# Remove bg of DEDUPED images only (cached, mirrors split/class layout)
# NOTE: default bria-rmbg (~1GB) OOMs here -> use tiny u2netp, one shared session
REM_BG_SESSION = new_session("u2netp")

def remove_bg(pil: Image.Image) -> Image.Image:
    fg = remove(pil.convert("RGB"), session=REM_BG_SESSION)
    bg = Image.new("RGB", fg.size, (255, 255, 255))
    bg.paste(fg, mask=fg.split()[3])
    return bg

def ensure_nobg(src_paths, src_root: Path):
    out = []
    for p in src_paths:
        q = NOBG_DIR / p.relative_to(src_root)
        if not q.exists():
            q.parent.mkdir(parents=True, exist_ok=True)
            remove_bg(Image.open(p)).save(q)
        out.append(q)
    return out

deduped_paths = sorted(
    p for p in DEDUPED_DIR.rglob("*") if p.suffix.lower() in IMAGE_EXTS
)
nobg_train = ensure_nobg(deduped_paths, DEDUPED_DIR)
print(f"bg-removed {len(nobg_train)} deduped images -> {NOBG_DIR}")

bg-removed 10730 deduped images -> /tmp/bananafp/nobg


In [10]:
# Re-embed bg-removed images, fit KMeans (final model), eval on valid/test
def labeled_paths(split: str):
    d = dataset / split
    paths, labels = [], []
    for class_dir in sorted(d.iterdir()):
        if not class_dir.is_dir():
            continue
        for p in sorted(class_dir.rglob("*")):
            if p.suffix.lower() in IMAGE_EXTS:
                paths.append(p)
                labels.append(class_dir.name)
    return paths, np.array(labels)

# train on deduped bg-removed re-embeddings
X_train = embed_paths(nobg_train)
y_train = np.array([p.parent.name for p in deduped_paths])

kmeans = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=0).fit(X_train)

# cluster -> label by majority vote
cluster2label = {}
for c in range(N_CLUSTERS):
    members = y_train[kmeans.labels_ == c]
    vals, counts = np.unique(members, return_counts=True)
    cluster2label[c] = vals[int(np.argmax(counts))]
print("cluster -> label:", cluster2label)

def evaluate(split: str):
    paths, y_true = labeled_paths(split)
    nobg = ensure_nobg(paths, dataset)
    X = embed_paths(nobg)
    y_pred = np.array([cluster2label[c] for c in kmeans.predict(X)])
    acc = float((y_pred == y_true).mean())
    print(f"{split}: acc={acc:.3f} (n={len(y_true)})")
    return acc

for s in ["valid", "test"]:
    evaluate(s)

cluster -> label: {0: np.str_('ripe'), 1: np.str_('rotten'), 2: np.str_('overripe'), 3: np.str_('unripe')}
valid: acc=0.643 (n=1123)
test: acc=0.610 (n=562)


In [11]:
from collections import defaultdict

rf_groups = defaultdict(list)
for p in image_paths:
    stem = p.name.split(".rf.")[0]
    rf_groups[stem].append(p)

print(len(image_paths), "->", len(rf_groups), "unique originals")

13478 -> 5616 unique originals


In [12]:
from collections import Counter
sizes = Counter(len(v) for v in rf_groups.values())
print(sorted(sizes.items()))  # e.g. [(1, 12), (3, 3930)] -> augmented 3x

[(1, 1685), (3, 3931)]


In [13]:
import re
from collections import defaultdict, Counter

UUID_RE = re.compile(r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}")

groups = defaultdict(list)
no_uuid = []
for p in image_paths:
    m = UUID_RE.search(p.name)
    if m:
        groups[m.group()].append(p)
    else:
        no_uuid.append(p)  # fall back to .rf. prefix for these

print(len(image_paths), "->", len(groups), "unique UUIDs,", len(no_uuid), "without UUID")
print(sorted(Counter(len(v) for v in groups.values()).items()))

13478 -> 5616 unique UUIDs, 0 without UUID
[(1, 1685), (3, 3931)]


In [14]:
emb = embed_paths(image_paths)
Z = emb / np.linalg.norm(emb, axis=1, keepdims=True)
idx_of = {p: i for i, p in enumerate(image_paths)}

sib_sims, other_sims = [], []
for u, members in groups.items():
    ids = [idx_of[p] for p in members]
    if len(ids) < 2:
        continue
    for i in ids:
        sims = Z @ Z[i]
        sims[i] = -1                             # ignore self
        mask = np.zeros(len(Z), bool); mask[ids] = True
        sib_sims.append(sims[mask].max())        # best sibling
        other_sims.append(sims[~mask].max())     # best non-sibling

sib_sims, other_sims = np.array(sib_sims), np.array(other_sims)
print("sibling > nearest outsider:", (sib_sims > other_sims).mean())

sibling > nearest outsider: 0.1890104299160519


In [15]:
copy_pairs = []
orphans = []
for u, members in groups.items():
    copies = [p for p in members if "copy" in p.name.lower()]
    origs  = [p for p in members if "copy" not in p.name.lower()]
    if not copies:
        continue
    if not origs:
        orphans.extend(copies)          # copy exists but original doesn't
        continue
    for c in copies:
        copy_pairs.append((c, origs))

print(len(copy_pairs), "copy files with an original,", len(orphans), "orphans")

0 copy files with an original, 893 orphans


## Is `-Copy` a duplicate?
Compare each `-Copy` file to its same-UUID original(s) vs. the nearest unrelated image, eyeball pairs, and check whether copies cross splits (leakage).

In [16]:
# similarity: copy vs its original(s) vs nearest unrelated image
sib, out = [], []
for c, origs in copy_pairs:
    i = idx_of[c]
    sims = Z @ Z[i]
    sims[i] = -1                                   # ignore self
    o_ids = [idx_of[p] for p in origs]
    sib.append(sims[o_ids].max())                  # best original
    mask = np.ones(len(Z), bool); mask[o_ids] = False; mask[i] = False
    out.append(sims[mask].max())                   # best unrelated image

sib, out = np.array(sib), np.array(out)
print("copy closest to its original:", (sib > out).mean())
print("median sim to original:", np.median(sib), "| to nearest other:", np.median(out))

copy closest to its original: nan
median sim to original: nan | to nearest other: nan


/tmp/ipykernel_173906/4201984946.py:13: RuntimeWarning: Mean of empty slice
  print("copy closest to its original:", (sib > out).mean())
/home/irzam/dev/college/kcvlbe/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/irzam/dev/college/kcvlbe/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3862: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,


In [17]:
# eyeball: copy vs original side by side
import random
import matplotlib.pyplot as plt

sample = random.sample(copy_pairs, min(8, len(copy_pairs)))
if sample:
    fig, axes = plt.subplots(len(sample), 2, figsize=(5, 2.5 * len(sample)), squeeze=False)
    for row, (c, origs) in zip(axes, sample):
        row[0].imshow(Image.open(c)); row[0].set_title("copy", fontsize=8)
        row[1].imshow(Image.open(origs[0])); row[1].set_title("original", fontsize=8)
        for ax in row:
            ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("no copy/original pairs to show")

no copy/original pairs to show


In [18]:
# leakage: do copies and their originals land in different splits?
split_of = lambda p: p.relative_to(dataset).parts[0]  # train / valid / test

cross = [(c, o) for c, origs in copy_pairs for o in origs if split_of(c) != split_of(o)]
print(len(cross), "copy/original pairs in different splits")
print(Counter((split_of(c), split_of(o)) for c, o in cross))

0 copy/original pairs in different splits
Counter()
